<a href="https://colab.research.google.com/github/miriamamin1213-ux/Seed42_Models/blob/main/GPT2_Old%20%26NewCN_%20Hancock_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -U torchao

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from transformers import (
    GPT2Tokenizer,
    GPT2Model
)

from peft import LoraConfig, get_peft_model, TaskType


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# ------------------------------------------------------------
# Read Data
# ------------------------------------------------------------

import gdown
import json
import pandas as pd

# Clinical data
gdown.download(
    id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
    output="clinical_data.json",
    quiet=False
)

# Pathological data
gdown.download(
    id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
    output="pathological_data.json",
    quiet=False
)

with open("clinical_data.json", "r") as f:
    clinical = pd.DataFrame(json.load(f))

with open("pathological_data.json", "r") as f:
    pathology = pd.DataFrame(json.load(f))

    df = clinical.merge(
    pathology,
    on='patient_id',
    how='inner'
)

print(df.shape)

print("Clinical shape:", clinical.shape)
print("Pathology shape:", pathology.shape)

# ============================================================
# HPV Labels
# ============================================================

df = df[
    df['hpv_association_p16'].isin(
        ['positive', 'negative']
    )
].copy()

df['HPV'] = df['hpv_association_p16'].map({
    'negative': 0,
    'positive': 1
})

# ============================================================
# GPT2 Hancock Features
# ============================================================

features = [
    'age_at_initial_diagnosis',
    'sex',
    'smoking_status',
    'primarily_metastasis',
    'first_treatment_intent',
    'first_treatment_modality',
    'days_to_first_treatment',
    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'histologic_type',
    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',
    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',
    'resection_status',
    'infiltration_depth_in_mm'
]

# ============================================================
# Missing Values
# ============================================================

for col in features:

    if df[col].dtype == object:

        df[col] = df[col].fillna(
            df[col].mode()[0]
        )

    else:

        df[col] = df[col].fillna(
            df[col].median()
        )

# ============================================================
# Narrative Function
# ============================================================

def patient_to_text(row):

    text = (
        f"This patient was diagnosed at age {row['age_at_initial_diagnosis']}. "
        f"Sex is {row['sex']}. "
        f"Smoking status is {row['smoking_status']}. "
        f"Primary tumour site is {row['primary_tumor_site']}. "
        f"Histologic type is {row['histologic_type']}. "
        f"Pathological T stage is {row['pT_stage']}. "
        f"Pathological N stage is {row['pN_stage']}. "
        f"The patient has "
        f"{row['number_of_positive_lymph_nodes']} positive lymph nodes and "
        f"{row['number_of_resected_lymph_nodes']} resected lymph nodes. "
        f"Perinodal invasion status is {row['perinodal_invasion']}. "
        f"Lymphovascular invasion status is {row['lymphovascular_invasion_L']}. "
        f"Vascular invasion status is {row['vascular_invasion_V']}. "
        f"Perineural invasion status is {row['perineural_invasion_Pn']}. "
        f"Resection status is {row['resection_status']}. "
        f"Infiltration depth is {row['infiltration_depth_in_mm']} mm. "
        f"First treatment intent was {row['first_treatment_intent']}. "
        f"First treatment modality was {row['first_treatment_modality']}. "
        f"Days to first treatment was {row['days_to_first_treatment']}. "
        f"Adjuvant treatment intent was {row['adjuvant_treatment_intent']}. "
        f"Adjuvant radiotherapy was {row['adjuvant_radiotherapy']}. "
        f"Adjuvant radiotherapy modality was {row['adjuvant_radiotherapy_modality']}. "
        f"Adjuvant systemic therapy was {row['adjuvant_systemic_therapy']}. "
        f"Adjuvant systemic therapy modality was {row['adjuvant_systemic_therapy_modality']}. "
        f"Adjuvant radiochemotherapy was {row['adjuvant_radiochemotherapy']}. "
        f"Predict whether the patient is HPV positive or HPV negative."
    )

    return text

df["patient_text"] = df.apply(
    patient_to_text,
    axis=1
)


# ============================================================
# Text and labels
# ============================================================

texts = df["patient_text"].tolist()

labels = (
    df["HPV"]
    .astype(int)
    .to_numpy()
)

# ============================================================
# Train-test split
# ============================================================

train_texts, test_texts, y_train, y_test = (
    train_test_split(
        texts,
        labels,
        test_size=0.2,
        random_state=SEED,
        stratify=labels
    )
)

print("\nTrain class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())


# ============================================================
# Tokenizer
# ============================================================

MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(
    MODEL_NAME
)

# GPT-2 has no default padding token
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 256


# ============================================================
# Dataset
# ============================================================

class HPVTextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=128
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        encoding = self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": (
                encoding["input_ids"].squeeze(0)
            ),
            "attention_mask": (
                encoding["attention_mask"].squeeze(0)
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long
            )
        }


train_dataset = HPVTextDataset(
    texts=train_texts,
    labels=y_train,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = HPVTextDataset(
    texts=test_texts,
    labels=y_test,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 4

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT-2 text classifier
# ============================================================

class HPVNetGPT2Text(nn.Module):

    def __init__(
        self,
        model_name="openai-community/gpt2",
        num_classes=2,
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.freeze_gpt2 = freeze_gpt2

        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = (
            tokenizer.pad_token_id
        )

        # -------------------------
        # LoRA
        # -------------------------
        if freeze_gpt2:

            lora_config = LoraConfig(
                r=6,
                lora_alpha=12,
                target_modules=["c_attn"],
                lora_dropout=0.0,
                bias="none",
                task_type=TaskType.FEATURE_EXTRACTION
            )

            self.gpt2 = get_peft_model(
                self.gpt2,
                lora_config
            )

        hidden_size = self.gpt2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        # if freeze_gpt2:
        #     self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Masked mean pooling
        expanded_mask = (
            attention_mask
            .unsqueeze(-1)
            .expand_as(hidden_states)
            .float()
        )

        hidden_sum = (
            hidden_states * expanded_mask
        ).sum(dim=1)

        valid_token_count = (
            expanded_mask.sum(dim=1)
            .clamp(min=1e-9)
        )

        pooled_output = (
            hidden_sum / valid_token_count
        )

        logits = self.classifier(
            pooled_output
        )

        return logits


# ============================================================
# Model
# ============================================================

model = HPVNetGPT2Text(
    model_name=MODEL_NAME,
    num_classes=2,
    freeze_gpt2=True
).to(device)


# ============================================================
# Class weights
# ============================================================

class_counts = np.bincount(y_train)

class_weights = (
    len(y_train)
    / (
        len(class_counts) *
        class_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("\nClass weights:", class_weights)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# Optimiser
# ============================================================

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)


# ============================================================
# Training
# ============================================================

EPOCHS = 100
best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        train_loss = criterion(
            outputs,
            labels
        )

        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter: parameter.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += (
            train_loss.item() *
            input_ids.size(0)
        )

        train_sample_count += (
            input_ids.size(0)
        )

    average_train_loss = (
        train_loss_sum /
        train_sample_count
    )

    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch[
                "input_ids"
            ].to(
                device,
                non_blocking=True
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            labels = batch["label"].to(
                device,
                non_blocking=True
            )

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            test_loss = criterion(
                outputs,
                labels
            )

            test_loss_sum += (
                test_loss.item() *
                input_ids.size(0)
            )

            test_sample_count += (
                input_ids.size(0)
            )

    average_test_loss = (
        test_loss_sum /
        test_sample_count
    )

    if average_test_loss < best_val_loss:

        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_text.pth"
        )

    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Load best model
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_text.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()


# ============================================================
# Inference
# ============================================================

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_probabilities.append(
            probabilities.cpu()
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            labels.cpu()

                    )


probabilities = torch.cat(
    all_probabilities,
    dim=0
).numpy()

predicted = torch.cat(
    all_predictions,
    dim=0
).numpy()

y_true = torch.cat(
    all_targets,
    dim=0
).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

bal_acc = balanced_accuracy_score(
    y_true,
    predicted
)

f1 = f1_score(
    y_true,
    predicted,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    probabilities[:, 1]
)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 125MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 112MB/s]


(763, 49)
Clinical shape: (763, 32)
Pathology shape: (763, 18)

Train class distribution:
0    152
1    113
Name: count, dtype: int64

Test class distribution:
0    39
1    28
Name: count, dtype: int64


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(



Class weights: tensor([0.8717, 1.1726], device='cuda:0')
Trainable parameters: 321410


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 005, Train=0.5695, Test=0.5732, Best Epoch=4
Epoch 010, Train=0.5307, Test=0.6254, Best Epoch=7
Epoch 015, Train=0.5048, Test=0.5135, Best Epoch=7
Epoch 020, Train=0.4881, Test=0.6594, Best Epoch=7
Epoch 025, Train=0.4856, Test=0.8163, Best Epoch=7
Epoch 030, Train=0.7056, Test=0.6972, Best Epoch=7
Epoch 035, Train=0.5445, Test=0.5443, Best Epoch=7
Epoch 040, Train=0.4749, Test=0.5968, Best Epoch=7
Epoch 045, Train=0.2935, Test=1.0165, Best Epoch=7
Epoch 050, Train=0.1806, Test=1.9852, Best Epoch=7
Epoch 055, Train=0.2293, Test=2.5637, Best Epoch=7
Epoch 060, Train=0.0325, Test=3.7712, Best Epoch=7
Epoch 065, Train=0.0331, Test=3.1748, Best Epoch=7
Epoch 070, Train=0.2390, Test=3.8294, Best Epoch=7
Epoch 075, Train=0.0634, Test=4.4587, Best Epoch=7
Epoch 080, Train=0.2294, Test=5.1918, Best Epoch=7
Epoch 085, Train=0.0374, Test=4.7012, Best Epoch=7
Epoch 090, Train=0.1214, Test=4.0698, Best Epoch=7
Epoch 095, Train=0.0332, Test=5.8942, Best Epoch=7
Epoch 100, Train=0.0719, Test=6

In [2]:
!pip install -U torchao

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from transformers import (
    GPT2Tokenizer,
    GPT2Model
)

from peft import LoraConfig, get_peft_model, TaskType


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# ------------------------------------------------------------
# Read Data
# ------------------------------------------------------------

import gdown
import json
import pandas as pd

# Clinical data
gdown.download(
    id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
    output="clinical_data.json",
    quiet=False
)

# Pathological data
gdown.download(
    id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
    output="pathological_data.json",
    quiet=False
)

with open("clinical_data.json", "r") as f:
    clinical = pd.DataFrame(json.load(f))

with open("pathological_data.json", "r") as f:
    pathology = pd.DataFrame(json.load(f))

    df = clinical.merge(
    pathology,
    on='patient_id',
    how='inner'
)

print(df.shape)

print("Clinical shape:", clinical.shape)
print("Pathology shape:", pathology.shape)

# ============================================================
# HPV Labels
# ============================================================

df = df[
    df['hpv_association_p16'].isin(
        ['positive', 'negative']
    )
].copy()

df['HPV'] = df['hpv_association_p16'].map({
    'negative': 0,
    'positive': 1
})

# ============================================================
# GPT2 Hancock Features
# ============================================================

features = [
    'age_at_initial_diagnosis',
    'sex',
    'smoking_status',
    'primarily_metastasis',
    'first_treatment_intent',
    'first_treatment_modality',
    'days_to_first_treatment',
    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'histologic_type',
    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',
    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',
    'resection_status',
    'infiltration_depth_in_mm'
]

# ============================================================
# Missing Values
# ============================================================

for col in features:

    if df[col].dtype == object:

        df[col] = df[col].fillna(
            df[col].mode()[0]
        )

    else:

        df[col] = df[col].fillna(
            df[col].median()
        )

# ============================================================
# Narrative Function
# ============================================================

def patient_to_text(row):

    text = (

        "You are a clinician specialising in head and neck cancer (HNC). "
        "Your task is to predict the patient's Human Papillomavirus (HPV) status "
        "using clinical, pathological, and treatment information. "
        "In head and neck cancer, pathological tumour stage (pT) reflects the local extent "
        "of the primary tumour, with higher stages indicating more advanced local disease. "
        "Pathological nodal stage (pN) reflects the degree of regional lymph node involvement. "
        "Resection status describes residual disease after surgery, where R0 indicates "
        "complete tumour removal with negative margins, R1 indicates microscopic residual "
        "tumour, and R2 indicates macroscopic residual tumour. "

        f"This patient was diagnosed at age {row['age_at_initial_diagnosis']} "
        f"and is {row['sex']} with a smoking status of {row['smoking_status']}. "

        f"The primary tumour site is {row['primary_tumor_site']} "
        f"and the histologic type is {row['histologic_type']}. "

        f"The pathological primary tumour stage is {row['pT_stage']} "
        f"and the pathological nodal stage is {row['pN_stage']}. "

        f"The patient has {row['number_of_positive_lymph_nodes']} positive lymph nodes "
        f"out of {row['number_of_resected_lymph_nodes']} resected lymph nodes. "

        f"Perinodal invasion is {row['perinodal_invasion']}. "
        f"Lymphovascular invasion is {row['lymphovascular_invasion_L']}. "
        f"Vascular invasion is {row['vascular_invasion_V']}. "
        f"Perineural invasion is {row['perineural_invasion_Pn']}. "

        f"Resection status is {row['resection_status']}. "
        f"Infiltration depth is {row['infiltration_depth_in_mm']} mm. "

        f"The first treatment was given with {row['first_treatment_intent']} intent "
        f"and consisted of {row['first_treatment_modality']}. "
        f"Treatment was initiated {row['days_to_first_treatment']} days after diagnosis. "

        f"Adjuvant treatment was given with {row['adjuvant_treatment_intent']} intent. "
        f"Adjuvant radiotherapy was {row['adjuvant_radiotherapy']} "
        f"using {row['adjuvant_radiotherapy_modality']}. "

        f"Adjuvant systemic therapy was {row['adjuvant_systemic_therapy']} "
        f"and the recorded systemic therapy modality was "
        f"{row['adjuvant_systemic_therapy_modality']}. "

        f"Adjuvant radiochemotherapy was "
        f"{row['adjuvant_radiochemotherapy']}. "

        "Based on the information provided, predict whether the patient is "
        "HPV positive or HPV negative."
    )

    return text

df["patient_text"] = df.apply(
    patient_to_text,
    axis=1
)

print(df.columns.tolist())


# ============================================================
# Text and labels
# ============================================================

texts = df["patient_text"].tolist()

labels = (
    df["HPV"]
    .astype(int)
    .to_numpy()
)

# ============================================================
# Train-test split
# ============================================================

train_texts, test_texts, y_train, y_test = (
    train_test_split(
        texts,
        labels,
        test_size=0.2,
        random_state=SEED,
        stratify=labels
    )
)

print("\nTrain class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())


# ============================================================
# Tokenizer
# ============================================================

MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(
    MODEL_NAME
)

# GPT-2 has no default padding token
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 128


# ============================================================
# Dataset
# ============================================================

class HPVTextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=128
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        encoding = self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": (
                encoding["input_ids"].squeeze(0)
            ),
            "attention_mask": (
                encoding["attention_mask"].squeeze(0)
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long
            )
        }


train_dataset = HPVTextDataset(
    texts=train_texts,
    labels=y_train,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = HPVTextDataset(
    texts=test_texts,
    labels=y_test,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 128

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT-2 text classifier
# ============================================================

class HPVNetGPT2Text(nn.Module):

    def __init__(
        self,
        model_name="openai-community/gpt2",
        num_classes=2,
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.freeze_gpt2 = freeze_gpt2

        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = (
            tokenizer.pad_token_id
        )

        # -------------------------
        # LoRA
        # -------------------------
        if freeze_gpt2:

            lora_config = LoraConfig(
                r=6,
                lora_alpha=12,
                target_modules=["c_attn"],
                lora_dropout=0.0,
                bias="none",
                task_type=TaskType.FEATURE_EXTRACTION
            )

            self.gpt2 = get_peft_model(
                self.gpt2,
                lora_config
            )

        hidden_size = self.gpt2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        # if freeze_gpt2:
        #     self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Masked mean pooling
        expanded_mask = (
            attention_mask
            .unsqueeze(-1)
            .expand_as(hidden_states)
            .float()
        )

        hidden_sum = (
            hidden_states * expanded_mask
        ).sum(dim=1)

        valid_token_count = (
            expanded_mask.sum(dim=1)
            .clamp(min=1e-9)
        )

        pooled_output = (
            hidden_sum / valid_token_count
        )

        logits = self.classifier(
            pooled_output
        )

        return logits


# ============================================================
# Model
# ============================================================

model = HPVNetGPT2Text(
    model_name=MODEL_NAME,
    num_classes=2,
    freeze_gpt2=True
).to(device)


# ============================================================
# Class weights
# ============================================================

class_counts = np.bincount(y_train)

class_weights = (
    len(y_train)
    / (
        len(class_counts) *
        class_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("\nClass weights:", class_weights)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# Optimiser
# ============================================================

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)


# ============================================================
# Training
# ============================================================

EPOCHS = 100
best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        train_loss = criterion(
            outputs,
            labels
        )

        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter: parameter.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += (
            train_loss.item() *
            input_ids.size(0)
        )

        train_sample_count += (
            input_ids.size(0)
        )

    average_train_loss = (
        train_loss_sum /
        train_sample_count
    )

    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch[
                "input_ids"
            ].to(
                device,
                non_blocking=True
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            labels = batch["label"].to(
                device,
                non_blocking=True
            )

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            test_loss = criterion(
                outputs,
                labels
            )

            test_loss_sum += (
                test_loss.item() *
                input_ids.size(0)
            )

            test_sample_count += (
                input_ids.size(0)
            )

    average_test_loss = (
        test_loss_sum /
        test_sample_count
    )

    if average_test_loss < best_val_loss:

        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_text.pth"
        )

    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Load best model
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_text.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()


# ============================================================
# Inference
# ============================================================

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_probabilities.append(
            probabilities.cpu()
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            labels.cpu()

                    )


probabilities = torch.cat(
    all_probabilities,
    dim=0
).numpy()

predicted = torch.cat(
    all_predictions,
    dim=0
).numpy()

y_true = torch.cat(
    all_targets,
    dim=0
).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

bal_acc = balanced_accuracy_score(
    y_true,
    predicted
)

f1 = f1_score(
    y_true,
    predicted,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    probabilities[:, 1]
)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 126MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 119MB/s]


(763, 49)
Clinical shape: (763, 32)
Pathology shape: (763, 18)
['patient_id', 'year_of_initial_diagnosis', 'age_at_initial_diagnosis', 'sex', 'smoking_status', 'primarily_metastasis', 'survival_status', 'survival_status_with_cause', 'days_to_last_information', 'first_treatment_intent', 'first_treatment_modality', 'days_to_first_treatment', 'adjuvant_treatment_intent', 'adjuvant_radiotherapy', 'adjuvant_radiotherapy_modality', 'adjuvant_systemic_therapy', 'adjuvant_systemic_therapy_modality', 'adjuvant_radiochemotherapy', 'recurrence', 'days_to_recurrence', 'progress_1', 'days_to_progress_1', 'progress_2', 'days_to_progress_2', 'metastasis_1_locations', 'days_to_metastasis_1', 'metastasis_2_locations', 'days_to_metastasis_2', 'metastasis_3_locations', 'days_to_metastasis_3', 'metastasis_4_locations', 'days_to_metastasis_4', 'primary_tumor_site', 'pT_stage', 'pN_stage', 'grading', 'hpv_association_p16', 'number_of_positive_lymph_nodes', 'number_of_resected_lymph_nodes', 'perinodal_invasi

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(



Class weights: tensor([0.8717, 1.1726], device='cuda:0')
Trainable parameters: 321410


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 005, Train=0.6972, Test=0.6954, Best Epoch=1
Epoch 010, Train=0.6908, Test=0.6985, Best Epoch=9
Epoch 015, Train=0.6905, Test=0.6954, Best Epoch=9
Epoch 020, Train=0.7001, Test=0.6948, Best Epoch=9
Epoch 025, Train=0.6926, Test=0.6930, Best Epoch=9
Epoch 030, Train=0.6954, Test=0.6953, Best Epoch=9
Epoch 035, Train=0.6958, Test=0.6936, Best Epoch=9
Epoch 040, Train=0.6974, Test=0.6930, Best Epoch=40
Epoch 045, Train=0.6997, Test=0.6991, Best Epoch=40
Epoch 050, Train=0.6997, Test=0.6932, Best Epoch=40
Epoch 055, Train=0.6966, Test=0.6948, Best Epoch=40
Epoch 060, Train=0.6936, Test=0.6942, Best Epoch=40
Epoch 065, Train=0.6964, Test=0.6931, Best Epoch=40
Epoch 070, Train=0.6923, Test=0.6931, Best Epoch=40
Epoch 075, Train=0.6936, Test=0.6930, Best Epoch=40
Epoch 080, Train=0.6917, Test=0.6939, Best Epoch=40
Epoch 085, Train=0.6938, Test=0.6930, Best Epoch=40
Epoch 090, Train=0.6932, Test=0.6931, Best Epoch=40
Epoch 095, Train=0.6943, Test=0.6930, Best Epoch=40
Epoch 100, Train=0.

GPT2 HANCOCK NEW NARRATIVE MAX LENGTH 512 + LORA

In [3]:
!pip install -U torchao

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from transformers import (
    GPT2Tokenizer,
    GPT2Model
)

from peft import LoraConfig, get_peft_model, TaskType


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# ------------------------------------------------------------
# Read Data
# ------------------------------------------------------------

import gdown
import json
import pandas as pd

# Clinical data
gdown.download(
    id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
    output="clinical_data.json",
    quiet=False
)

# Pathological data
gdown.download(
    id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
    output="pathological_data.json",
    quiet=False
)

with open("clinical_data.json", "r") as f:
    clinical = pd.DataFrame(json.load(f))

with open("pathological_data.json", "r") as f:
    pathology = pd.DataFrame(json.load(f))

    df = clinical.merge(
    pathology,
    on='patient_id',
    how='inner'
)

print(df.shape)

print("Clinical shape:", clinical.shape)
print("Pathology shape:", pathology.shape)

# ============================================================
# HPV Labels
# ============================================================

df = df[
    df['hpv_association_p16'].isin(
        ['positive', 'negative']
    )
].copy()

df['HPV'] = df['hpv_association_p16'].map({
    'negative': 0,
    'positive': 1
})

# ============================================================
# GPT2 Hancock Features
# ============================================================

features = [
    'age_at_initial_diagnosis',
    'sex',
    'smoking_status',
    'primarily_metastasis',
    'first_treatment_intent',
    'first_treatment_modality',
    'days_to_first_treatment',
    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'histologic_type',
    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',
    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',
    'resection_status',
    'infiltration_depth_in_mm'
]

# ============================================================
# Missing Values
# ============================================================

for col in features:

    if df[col].dtype == object:

        df[col] = df[col].fillna(
            df[col].mode()[0]
        )

    else:

        df[col] = df[col].fillna(
            df[col].median()
        )

# ============================================================
# Narrative Function
# ============================================================

def patient_to_text(row):

    text = (

        "You are a clinician specialising in head and neck cancer (HNC). "
        "Your task is to predict the patient's Human Papillomavirus (HPV) status "
        "using clinical, pathological, and treatment information. "
        "In head and neck cancer, pathological tumour stage (pT) reflects the local extent "
        "of the primary tumour, with higher stages indicating more advanced local disease. "
        "Pathological nodal stage (pN) reflects the degree of regional lymph node involvement. "
        "Resection status describes residual disease after surgery, where R0 indicates "
        "complete tumour removal with negative margins, R1 indicates microscopic residual "
        "tumour, and R2 indicates macroscopic residual tumour. "

        f"This patient was diagnosed at age {row['age_at_initial_diagnosis']} "
        f"and is {row['sex']} with a smoking status of {row['smoking_status']}. "

        f"The primary tumour site is {row['primary_tumor_site']} "
        f"and the histologic type is {row['histologic_type']}. "

        f"The pathological primary tumour stage is {row['pT_stage']} "
        f"and the pathological nodal stage is {row['pN_stage']}. "

        f"The patient has {row['number_of_positive_lymph_nodes']} positive lymph nodes "
        f"out of {row['number_of_resected_lymph_nodes']} resected lymph nodes. "

        f"Perinodal invasion is {row['perinodal_invasion']}. "
        f"Lymphovascular invasion is {row['lymphovascular_invasion_L']}. "
        f"Vascular invasion is {row['vascular_invasion_V']}. "
        f"Perineural invasion is {row['perineural_invasion_Pn']}. "

        f"Resection status is {row['resection_status']}. "
        f"Infiltration depth is {row['infiltration_depth_in_mm']} mm. "

        f"The first treatment was given with {row['first_treatment_intent']} intent "
        f"and consisted of {row['first_treatment_modality']}. "
        f"Treatment was initiated {row['days_to_first_treatment']} days after diagnosis. "

        f"Adjuvant treatment was given with {row['adjuvant_treatment_intent']} intent. "
        f"Adjuvant radiotherapy was {row['adjuvant_radiotherapy']} "
        f"using {row['adjuvant_radiotherapy_modality']}. "

        f"Adjuvant systemic therapy was {row['adjuvant_systemic_therapy']} "
        f"and the recorded systemic therapy modality was "
        f"{row['adjuvant_systemic_therapy_modality']}. "

        f"Adjuvant radiochemotherapy was "
        f"{row['adjuvant_radiochemotherapy']}. "

        "Based on the information provided, predict whether the patient is "
        "HPV positive or HPV negative."
    )

    return text

df["patient_text"] = df.apply(
    patient_to_text,
    axis=1
)

print(df.columns.tolist())


# ============================================================
# Text and labels
# ============================================================

texts = df["patient_text"].tolist()

labels = (
    df["HPV"]
    .astype(int)
    .to_numpy()
)

# ============================================================
# Train-test split
# ============================================================

train_texts, test_texts, y_train, y_test = (
    train_test_split(
        texts,
        labels,
        test_size=0.2,
        random_state=SEED,
        stratify=labels
    )
)

print("\nTrain class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())


# ============================================================
# Tokenizer
# ============================================================

MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(
    MODEL_NAME
)

# GPT-2 has no default padding token
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 384


# ============================================================
# Dataset
# ============================================================

class HPVTextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=128
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        encoding = self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": (
                encoding["input_ids"].squeeze(0)
            ),
            "attention_mask": (
                encoding["attention_mask"].squeeze(0)
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long
            )
        }


train_dataset = HPVTextDataset(
    texts=train_texts,
    labels=y_train,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = HPVTextDataset(
    texts=test_texts,
    labels=y_test,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 8

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT-2 text classifier
# ============================================================

class HPVNetGPT2Text(nn.Module):

    def __init__(
        self,
        model_name="openai-community/gpt2",
        num_classes=2,
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.freeze_gpt2 = freeze_gpt2

        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = (
            tokenizer.pad_token_id
        )

        # -------------------------
        # LoRA
        # -------------------------
        if freeze_gpt2:

            lora_config = LoraConfig(
                r=6,
                lora_alpha=12,
                target_modules=["c_attn"],
                lora_dropout=0.0,
                bias="none",
                task_type=TaskType.FEATURE_EXTRACTION
            )

            self.gpt2 = get_peft_model(
                self.gpt2,
                lora_config
            )

        hidden_size = self.gpt2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        # if freeze_gpt2:
        #     self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Masked mean pooling
        expanded_mask = (
            attention_mask
            .unsqueeze(-1)
            .expand_as(hidden_states)
            .float()
        )

        hidden_sum = (
            hidden_states * expanded_mask
        ).sum(dim=1)

        valid_token_count = (
            expanded_mask.sum(dim=1)
            .clamp(min=1e-9)
        )

        pooled_output = (
            hidden_sum / valid_token_count
        )

        logits = self.classifier(
            pooled_output
        )

        return logits


# ============================================================
# Model
# ============================================================

model = HPVNetGPT2Text(
    model_name=MODEL_NAME,
    num_classes=2,
    freeze_gpt2=True
).to(device)


# ============================================================
# Class weights
# ============================================================

class_counts = np.bincount(y_train)

class_weights = (
    len(y_train)
    / (
        len(class_counts) *
        class_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("\nClass weights:", class_weights)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# Optimiser
# ============================================================

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)


# ============================================================
# Training
# ============================================================

EPOCHS = 100
best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        train_loss = criterion(
            outputs,
            labels
        )

        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter: parameter.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += (
            train_loss.item() *
            input_ids.size(0)
        )

        train_sample_count += (
            input_ids.size(0)
        )

    average_train_loss = (
        train_loss_sum /
        train_sample_count
    )

    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch[
                "input_ids"
            ].to(
                device,
                non_blocking=True
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            labels = batch["label"].to(
                device,
                non_blocking=True
            )

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            test_loss = criterion(
                outputs,
                labels
            )

            test_loss_sum += (
                test_loss.item() *
                input_ids.size(0)
            )

            test_sample_count += (
                input_ids.size(0)
            )

    average_test_loss = (
        test_loss_sum /
        test_sample_count
    )

    if average_test_loss < best_val_loss:

        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_text.pth"
        )

    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Load best model
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_text.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()


# ============================================================
# Inference
# ============================================================

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_probabilities.append(
            probabilities.cpu()
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            labels.cpu()

                    )


probabilities = torch.cat(
    all_probabilities,
    dim=0
).numpy()

predicted = torch.cat(
    all_predictions,
    dim=0
).numpy()

y_true = torch.cat(
    all_targets,
    dim=0
).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

bal_acc = balanced_accuracy_score(
    y_true,
    predicted
)

f1 = f1_score(
    y_true,
    predicted,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    probabilities[:, 1]
)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 137MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 134MB/s]


(763, 49)
Clinical shape: (763, 32)
Pathology shape: (763, 18)
['patient_id', 'year_of_initial_diagnosis', 'age_at_initial_diagnosis', 'sex', 'smoking_status', 'primarily_metastasis', 'survival_status', 'survival_status_with_cause', 'days_to_last_information', 'first_treatment_intent', 'first_treatment_modality', 'days_to_first_treatment', 'adjuvant_treatment_intent', 'adjuvant_radiotherapy', 'adjuvant_radiotherapy_modality', 'adjuvant_systemic_therapy', 'adjuvant_systemic_therapy_modality', 'adjuvant_radiochemotherapy', 'recurrence', 'days_to_recurrence', 'progress_1', 'days_to_progress_1', 'progress_2', 'days_to_progress_2', 'metastasis_1_locations', 'days_to_metastasis_1', 'metastasis_2_locations', 'days_to_metastasis_2', 'metastasis_3_locations', 'days_to_metastasis_3', 'metastasis_4_locations', 'days_to_metastasis_4', 'primary_tumor_site', 'pT_stage', 'pN_stage', 'grading', 'hpv_association_p16', 'number_of_positive_lymph_nodes', 'number_of_resected_lymph_nodes', 'perinodal_invasi

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(



Class weights: tensor([0.8717, 1.1726], device='cuda:0')
Trainable parameters: 321410


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 005, Train=0.6546, Test=0.6237, Best Epoch=5
Epoch 010, Train=0.5924, Test=0.5140, Best Epoch=8
Epoch 015, Train=0.5327, Test=0.4888, Best Epoch=12
Epoch 020, Train=0.4912, Test=0.5566, Best Epoch=12
Epoch 025, Train=0.2937, Test=0.7299, Best Epoch=12
Epoch 030, Train=0.1686, Test=1.3281, Best Epoch=12
Epoch 035, Train=0.1068, Test=1.9972, Best Epoch=12
Epoch 040, Train=0.0093, Test=2.6450, Best Epoch=12
Epoch 045, Train=0.0001, Test=2.7573, Best Epoch=12
Epoch 050, Train=0.0001, Test=2.8763, Best Epoch=12
Epoch 055, Train=0.0000, Test=3.0065, Best Epoch=12
Epoch 060, Train=0.0000, Test=3.0622, Best Epoch=12
Epoch 065, Train=0.0000, Test=3.1212, Best Epoch=12
Epoch 070, Train=0.0000, Test=3.1555, Best Epoch=12
Epoch 075, Train=0.0000, Test=3.2001, Best Epoch=12
Epoch 080, Train=0.0000, Test=3.2451, Best Epoch=12
Epoch 085, Train=0.0000, Test=3.2675, Best Epoch=12
Epoch 090, Train=0.0000, Test=3.3128, Best Epoch=12
Epoch 095, Train=0.0000, Test=3.3637, Best Epoch=12
Epoch 100, Tra